# Transcript Difficulty Analysis

For each person, sends their verbatim transcript to Claude and asks it to identify which of the 45 pairwise decisions showed signs of conflict or difficulty — based purely on speech patterns.

**Setup:** put your API key in `412study/.env`:
```
ANTHROPIC_API_KEY=sk-ant-...
```
The `.env` file is gitignored and never needs to be exported in the terminal.

In [1]:
import sys
!{sys.executable} -m pip install anthropic pdfplumber python-dotenv --quiet

import anthropic
import pdfplumber
import pandas as pd
import json
import os
import glob
from pydantic import BaseModel
from typing import Optional, List
from dotenv import load_dotenv

load_dotenv()  # loads ANTHROPIC_API_KEY from .env in the current directory

assert os.environ.get('ANTHROPIC_API_KEY'), "ANTHROPIC_API_KEY not found — check your .env file"
client = anthropic.Anthropic()
print('Client ready.')

Client ready.


In [2]:
# Pydantic schema for the structured output
class Decision(BaseModel):
    q: int                        # question number 1-45
    choice: str                   # 'A', 'B', or 'unclear'
    conflicted: bool              # True if signs of difficulty on this specific decision
    difficulty_score: int         # 1 (instant) to 5 (very hard)
    evidence: Optional[str]       # short quote showing conflict, or None

class TranscriptAnalysis(BaseModel):
    participant_speaker: str      # 'SPEAKER 1' or 'SPEAKER 2'
    decisions: List[Decision]

In [3]:
SYSTEM_PROMPT = """You are analyzing a verbatim audio transcript of a research study session.
In the study, a participant made exactly 45 pairwise comparison decisions, choosing between
Organization A and Organization B to receive a food donation. The participant spoke their
reasoning aloud while a researcher facilitated.

Your job:
1. Identify which speaker is the PARTICIPANT (the other speaker is the researcher who
   explains the task and asks follow-up questions).
2. Find each of the 45 decisions in order. The 1st decision made = question 1, 2nd = question 2, etc.
3. For each decision output:
   - q: question number (1-45)
   - choice: 'A', 'B', or 'unclear' (only 'unclear' if genuinely unresolvable)
   - conflicted: true ONLY when there is clear evidence of genuine uncertainty or difficulty
     on THAT specific decision
   - difficulty_score: 1 (instant, no hesitation) to 5 (prolonged deliberation or explicit struggle)
   - evidence: a verbatim quote of under 30 words from the transcript showing the conflict,
     or null if not conflicted

Signs of conflict/difficulty to look for:
- Explicit statements: 'this is hard', 'tough one', 'I'm not sure', 'it's close', 'tricky'
- Self-correction or changing their mind mid-sentence
- Extended back-and-forth reasoning before deciding (comparing both options multiple times)
- Asking the researcher for clarification because they cannot decide
- Long pauses or filler sounds ('hmm', 'uh', 'um') directly before stating the choice
- Hedging words immediately before the choice ('I think maybe', 'probably', 'I guess')

Important: most decisions will NOT be conflicted. Only mark conflicted=true when the transcript
shows clear evidence of genuine uncertainty on that specific decision — not just the person's
general thinking style. difficulty_score should reflect the effort on that decision specifically.

If the transcript has fewer than 45 identifiable decisions, output as many as you can find."""

print('System prompt ready.')

System prompt ready.


In [4]:
def extract_transcript(pdf_path: str) -> str:
    """Extract the transcript section from a PDF."""
    with pdfplumber.open(pdf_path) as pdf:
        text = '\n'.join(page.extract_text() or '' for page in pdf.pages)
    idx = text.lower().find('transcript')
    if idx == -1:
        return text  # fallback: use full text
    return text[idx:]


def analyze_transcript(person_id: str, transcript: str) -> TranscriptAnalysis:
    """Send transcript to Claude and get structured difficulty analysis."""
    response = client.messages.parse(
        model='claude-opus-4-6',
        max_tokens=8000,
        system=[
            {
                'type': 'text',
                'text': SYSTEM_PROMPT,
                'cache_control': {'type': 'ephemeral'}  # cache system prompt across all calls
            }
        ],
        messages=[
            {
                'role': 'user',
                'content': f'Participant ID: {person_id}\n\n{transcript}'
            }
        ],
        output_format=TranscriptAnalysis,
    )
    return response.parsed_output


print('Functions defined.')

Functions defined.


In [5]:
# Find all transcript PDFs
pdf_files = sorted(glob.glob('*_2026-01-28.pdf') + glob.glob('*-2026-01-28.pdf'))
print(f'Found {len(pdf_files)} transcript PDFs:')
for f in pdf_files:
    print(f'  {f}')

Found 15 transcript PDFs:
  D2_2026-01-28.pdf
  F1_2026-01-28.pdf
  F2_2026-01-28.pdf
  F3_2026-01-28.pdf
  R1_2026-01-28.pdf
  R2_2026-01-28.pdf
  R3_2026-01-28.pdf
  R5_2026-01-28.pdf
  R7_2026-01-28.pdf
  R8_2026-01-28.pdf
  V1_2026-01-28.pdf
  V3_2026-01-28.pdf
  V5_2026-01-28.pdf
  V6-2026-01-28.pdf
  V7_2026-01-28.pdf


In [6]:
# Run analysis for all persons
# This makes 15 API calls — takes ~2-3 minutes total
all_results = {}
failed = []

for pdf_path in pdf_files:
    # Extract person ID from filename (e.g. 'D2_2026-01-28.pdf' -> 'D2')
    person_id = os.path.basename(pdf_path).split('_')[0].split('-')[0]
    print(f'Analyzing {person_id}...', end=' ', flush=True)
    try:
        transcript = extract_transcript(pdf_path)
        analysis = analyze_transcript(person_id, transcript)
        all_results[person_id] = analysis
        n_conflicted = sum(1 for d in analysis.decisions if d.conflicted)
        print(f'done. {len(analysis.decisions)} decisions, {n_conflicted} conflicted.')
    except Exception as e:
        print(f'FAILED: {e}')
        failed.append(person_id)

print(f'\nComplete. {len(all_results)} succeeded, {len(failed)} failed.')
if failed:
    print(f'Failed: {failed}')

Analyzing D2... done. 45 decisions, 2 conflicted.
Analyzing F1... done. 15 decisions, 6 conflicted.
Analyzing F2... done. 19 decisions, 5 conflicted.
Analyzing F3... done. 31 decisions, 7 conflicted.
Analyzing R1... done. 17 decisions, 6 conflicted.
Analyzing R2... done. 20 decisions, 7 conflicted.
Analyzing R3... done. 45 decisions, 1 conflicted.
Analyzing R5... done. 45 decisions, 6 conflicted.
Analyzing R7... done. 45 decisions, 5 conflicted.
Analyzing R8... done. 45 decisions, 9 conflicted.
Analyzing V1... done. 32 decisions, 10 conflicted.
Analyzing V3... done. 45 decisions, 11 conflicted.
Analyzing V5... done. 45 decisions, 17 conflicted.
Analyzing V6... done. 45 decisions, 10 conflicted.
Analyzing V7... done. 45 decisions, 38 conflicted.

Complete. 15 succeeded, 0 failed.


In [7]:
# Flatten to a DataFrame
rows = []
for person_id, analysis in all_results.items():
    for d in analysis.decisions:
        rows.append({
            'personID':           person_id,
            'participant_speaker': analysis.participant_speaker,
            'num_Q':              d.q,
            'choice':             d.choice,
            'conflicted':         d.conflicted,
            'difficulty_score':   d.difficulty_score,
            'evidence':           d.evidence,
        })

difficulty_df = pd.DataFrame(rows)
print(difficulty_df.shape)
difficulty_df.head(10)

(539, 7)


,personID,participant_speaker,num_Q,choice,conflicted,difficulty_score,evidence
0,D2,SPEAKER 2,1,B,False,2,None
1,D2,SPEAKER 2,2,B,False,1,None
2,D2,SPEAKER 2,3,B,False,1,None
3,D2,SPEAKER 2,4,A,False,1,None
4,D2,SPEAKER 2,5,B,False,1,None
5,D2,SPEAKER 2,6,B,False,1,None
6,D2,SPEAKER 2,7,B,False,1,None
7,D2,SPEAKER 2,8,B,False,1,None
8,D2,SPEAKER 2,9,B,False,1,None
9,D2,SPEAKER 2,10,B,False,1,None


In [8]:
# Summary: conflicted decisions per person
summary = difficulty_df.groupby('personID').agg(
    total_decisions=('num_Q', 'count'),
    n_conflicted=('conflicted', 'sum'),
    avg_difficulty=('difficulty_score', 'mean'),
    max_difficulty=('difficulty_score', 'max'),
).assign(pct_conflicted=lambda d: d['n_conflicted'] / d['total_decisions'] * 100)
print(summary.sort_values('pct_conflicted', ascending=False).to_string())

          total_decisions  n_conflicted  avg_difficulty  max_difficulty  pct_conflicted
personID                                                                               
V7                     45            38        3.622222               4       84.444444
F1                     15             6        2.466667               4       40.000000
V5                     45            17        2.422222               5       37.777778
R1                     17             6        2.823529               5       35.294118
R2                     20             7        2.200000               5       35.000000
V1                     32            10        2.531250               5       31.250000
F2                     19             5        2.105263               4       26.315789
V3                     45            11        2.066667               5       24.444444
F3                     31             7        2.419355               5       22.580645
V6                     45       

In [9]:
# Show all conflicted decisions with evidence
conflicted = difficulty_df[difficulty_df['conflicted']].copy()
print(f'Total conflicted decisions: {len(conflicted)} across {conflicted["personID"].nunique()} persons\n')
for person in sorted(conflicted['personID'].unique()):
    p = conflicted[conflicted['personID'] == person]
    print(f'\n===== {person} ({len(p)} conflicted) =====')
    for _, row in p.iterrows():
        print(f'  Q{row["num_Q"]:2d} [diff={row["difficulty_score"]}] chose {row["choice"]}  |  {row["evidence"]}')

Total conflicted decisions: 140 across 15 persons


===== D2 (2 conflicted) =====
  Q25 [diff=2] chose A  |  Because of the organization's size and level of food access.
  Q33 [diff=3] chose B  |  B, because of the level of food access of coffee. Right? Maybe because of the level of food access and the organization size. Well, I guess those are the same.

===== F1 (6 conflicted) =====
  Q 3 [diff=3] chose A  |  Well, it's kinda weird because how we don't really serve organizations that are not in poverty. Mhmm. So I don't know
  Q 4 [diff=4] chose unclear  |  Yes. I would go with this one just because of the travel time. Okay. So I would give it to here.
  Q 6 [diff=3] chose A  |  I'll do it to here. Yeah. So I would get it here because of the poverty rate, and this is really long.
  Q 7 [diff=4] chose B  |  Yeah. Yeah. I would still give it To a. B this Travel time.
  Q 8 [diff=3] chose B  |  As soon as it hits an hour travel time, it it barely get volunteers Yeah. For Guess I would g

In [10]:
# Save to CSV
difficulty_df.to_csv('transcript_difficulty.csv', index=False)
print('Saved transcript_difficulty.csv')

Saved transcript_difficulty.csv
